In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import time
import joblib
from sklearn.impute import SimpleImputer

In [19]:
df = pd.read_csv("data_teacher_salaries.csv")

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Load the data
df = pd.read_csv("data_teacher_salaries.csv")
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
df.head()

Dataset shape: (12535, 32)

Column names:
['Unnamed: 0.1', 'Unnamed: 0', 'DISTRICT_NUMBER', 'DISTRICT', 'CAMPUS_NUMBER', 'CAMPUS', 'REGION', 'COUNTY', 'SCHOOL_TYPE', 'GRADES_SERVED', 'CHARTER', 'URBANICITY', 'NUMBER_OF_STUDENTS', 'OVERALL_RATING', 'OVERALL_SCORE', '%_ECONOMICALLY_DISADVANTAGED', '%_EB/EL_STUDENTS', 'FREE_LUNCH_QUALIFIED_COUNT', 'REDUCED-PRICE_LUNCH_QUALIFIED_COUNT', 'RACE_AMERICAN_INDIAN_OR_ALASKA_NATIVE', 'RACE_ASIAN', 'RACE_BLACK_OR_AFRICAN_AMERICAN', 'RACE_HISPANIC/LATINO', 'RACE_NATIVE_HAWAIIAN_OR_OTHER_PACIFIC_ISLANDER', 'RACE_TWO_OR_MORE_RACES', 'RACE_WHITE', 'TEACHERS_COUNT', 'LAT', 'LONG', 'MEDIAN_HOUSEHOLD_INCOME_AREA', 'ST_SCHID', 'AVG_TEACHER_SALARY']


,Unnamed: 0.1,Unnamed: 0,DISTRICT_NUMBER,DISTRICT,CAMPUS_NUMBER,CAMPUS,REGION,COUNTY,SCHOOL_TYPE,GRADES_SERVED,...,RACE_HISPANIC/LATINO,RACE_NATIVE_HAWAIIAN_OR_OTHER_PACIFIC_ISLANDER,RACE_TWO_OR_MORE_RACES,RACE_WHITE,TEACHERS_COUNT,LAT,LONG,MEDIAN_HOUSEHOLD_INCOME_AREA,ST_SCHID,AVG_TEACHER_SALARY
0,0,0,1902,CAYUGA ISD,1902001.0,CAYUGA H S,REGION 07: KILGORE,ANDERSON,High School,09 - 12,...,36.0,0.0,16.0,276.0,19.34,31.9569,-95.9744,72483.0,TX-001902-001902001,55570.0
1,1,1,1902,CAYUGA ISD,1902041.0,CAYUGA MIDDLE,REGION 07: KILGORE,ANDERSON,Middle School,06 - 08,...,32.0,0.0,18.0,264.0,11.20,31.9569,-95.9744,72483.0,TX-001902-001902041,55570.0
2,2,2,1902,CAYUGA ISD,1902103.0,CAYUGA EL,REGION 07: KILGORE,ANDERSON,Elementary,PK - 05,...,44.0,0.0,20.0,420.0,17.86,31.9569,-95.9744,72483.0,TX-001902-001902103,55570.0
3,3,3,1903,ELKHART ISD,1903001.0,ELKHART H S,REGION 07: KILGORE,ANDERSON,High School,09 - 12,...,76.0,2.0,20.0,502.0,35.72,31.6275,-95.5785,76397.0,TX-001903-001903001,47916.0
4,4,4,1903,ELKHART ISD,1903041.0,ELKHART MIDDLE,REGION 07: KILGORE,ANDERSON,Middle School,06 - 08,...,62.0,0.0,16.0,442.0,23.82,31.6279,-95.5780,76397.0,TX-001903-001903041,47916.0


In [21]:
def preprocess_data(df):
    df_processed = df.copy()

    # Drop unnecessary identifier columns
    df_processed = df_processed.drop(['Unnamed: 0.1', 'Unnamed: 0'], axis=1, errors='ignore')

    id_columns = ['ST_SCHID', 'DISTRICT_NUMBER', 'CAMPUS_NUMBER', 'DISTRICT', 'CAMPUS']
    df_processed = df_processed.drop(columns=[col for col in id_columns if col in df_processed.columns], errors='ignore')

    # Separate features and target
    target_col = 'AVG_TEACHER_SALARY'
    X = df_processed.drop(target_col, axis=1)
    y = df_processed[target_col]

    # Remove any rows where target is NaN, infinity, or extremely large
    print(f"Original target range: {y.min()} to {y.max()}")
    print(f"Target NaN count: {y.isna().sum()}")
    print(f"Target infinite count: {np.isinf(y).sum()}")

    # Remove problematic target values
    mask = ~(y.isna() | np.isinf(y) | (y > 1e10) | (y < 0))  # Remove NaN, inf, very large, negative
    X = X[mask]
    y = y[mask]

    print(f"After cleaning - Target range: {y.min()} to {y.max()}")
    print(f"Remaining samples: {len(y)}")

    # Handle missing values in features
    numeric_columns = X.select_dtypes(include=[np.number]).columns
    categorical_columns = X.select_dtypes(include=['object']).columns

    # Impute numeric columns with median
    imputer_numeric = SimpleImputer(strategy='median')
    X[numeric_columns] = imputer_numeric.fit_transform(X[numeric_columns])

    # Encode categorical variables
    label_encoders = {}
    for col in categorical_columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le

    return X, y, label_encoders

X, y, encoders = preprocess_data(df)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Target statistics:", y.describe())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

Original target range: 36081.0 to 110567.0
Target NaN count: 19
Target infinite count: 0
After cleaning - Target range: 36081.0 to 110567.0
Remaining samples: 12516
Features shape: (12516, 24)
Target shape: (12516,)
Target statistics: count     12516.000000
mean      57755.647172
std        5034.934723
min       36081.000000
25%       53993.000000
50%       58215.000000
75%       61811.000000
max      110567.000000
Name: AVG_TEACHER_SALARY, dtype: float64
Training set shape: (10012, 24)
Test set shape: (2504, 24)


In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

Training set shape: (10012, 24)
Test set shape: (2504, 24)


In [23]:
xgb_basic = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metric='rmse'
)

xgb_basic.fit(X_train, y_train)

xgb_param_grid = {
    'max_depth': [3, 6],
    'learning_rate': [0.05, 0.1]
}

xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    random_state=42,
    eval_metric='rmse'
)

xgb_grid_search = GridSearchCV(
    xgb_model,
    xgb_param_grid,
    cv=3,  # Reduce CV folds to 3 to speed up and reduce memory usage
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

xgb_grid_search.fit(X_train, y_train)

xgb_best = xgb_grid_search.best_estimator_
print("Best XGBoost parameters:", xgb_grid_search.best_params_)

Fitting 3 folds for each of 4 candidates, totalling 12 fits


KeyboardInterrupt: 

In [ ]:
# Make predictions
y_pred = xgb_best.predict(X_test)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nXGBoost Model Results:")
print(f"Test RMSE: ${rmse:,.2f}")
print(f"Test MAE: ${mae:,.2f}")
print(f"Test R²: {r2:.4f}")

# Cross-validation results
cv_scores = cross_val_score(xgb_best, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)
print(f"\nCross-Validation RMSE: {cv_rmse.mean():.2f} (+/- {cv_rmse.std() * 2:.2f})")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_best.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 10 Most Important Features:")
print(feature_importance.head(10))

In [ ]:
import time

# Measure training time
start_time = time.time()
xgb_final = xgb.XGBRegressor(**xgb_grid_search.best_params_, random_state=42)
xgb_final.fit(X_train, y_train)
training_time = time.time() - start_time

# Count hyperparameter trials
total_trials = len(xgb_grid_search.cv_results_['params'])

print(f"\nComputation Metrics:")
print(f"Training time: {training_time:.2f} seconds")
print(f"Total hyperparameter trials: {total_trials}")
print(f"Best CV RMSE: {np.sqrt(-xgb_grid_search.best_score_):.2f}")

In [ ]:
# Visualization
plt.figure(figsize=(12, 4))

# Plot 1: Actual vs Predicted
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.title(f'Actual vs Predicted Salaries\nR² = {r2:.4f}')

# Plot 2: Top Feature Importance
plt.subplot(1, 2, 2)
top_features = feature_importance.head(10)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 10 Feature Importances')

plt.tight_layout()
plt.show()

print(f"- Achieves R² of {r2:.4f}, explaining {r2*100:.1f}% of variance in salaries")
print(f"- RMSE of ${rmse:,.2f} represents reasonable prediction accuracy")

In [ ]:
from sklearn.linear_model import ElasticNetCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


# Create a pipeline that scales data and fits ElasticNet
elastic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("elasticnet", ElasticNetCV(
        l1_ratio=[.1, .5, .7, .9, 1.0],  # balance between L1 and L2 regularization
        alphas=[0.001, 0.01, 0.1, 1, 10],  # range of alpha values
        cv=5,
        random_state=42
    ))
])


In [ ]:
import time
from codecarbon import EmissionsTracker

start_time = time.time()
elastic_pipeline.fit(X_train, y_train)
training_time_en = time.time() - start_time

best_l1_ratio = elastic_pipeline.named_steps["elasticnet"].l1_ratio_
best_alpha = elastic_pipeline.named_steps["elasticnet"].alpha_

print(f"Best l1_ratio: {best_l1_ratio}")
print(f"Best alpha: {best_alpha}")
print(f"Training time: {training_time_en:.2f} seconds")

tracker = EmissionsTracker(project_name="ElasticNet_Training")
tracker.start()

elastic_pipeline.fit(X_train, y_train) # Re-fit to track emissions for this specific training run

training_emissions = tracker.stop()

print(f"Energy consumed: {tracker.final_emissions_data.energy_consumed:.6f} kWh")
print(f"CO₂ emissions: {training_emissions:.6f} kg CO₂eq")

tracker_pred = EmissionsTracker(project_name="ElasticNet_Prediction")
tracker_pred.start()

_ = elastic_pipeline.predict(X_test)

prediction_emissions = tracker_pred.stop()

# Estimate per 1000 examples
energy_per_1000 = tracker_pred.final_emissions_data.energy_consumed * (1000 / len(X_test))
co2_per_1000 = prediction_emissions * (1000 / len(X_test))

print(f"Energy consumed: {energy_per_1000:.8f} kWh")
print(f"CO₂ emissions: {co2_per_1000:.8f} kg CO₂eq")

In [ ]:
display(feature_importance)

In [ ]:
start_time = time.time()
elastic_pipeline.fit(X_train, y_train)
training_time_en = time.time() - start_time

best_l1_ratio = elastic_pipeline.named_steps["elasticnet"].l1_ratio_
best_alpha = elastic_pipeline.named_steps["elasticnet"].alpha_

print(f"Best l1_ratio: {best_l1_ratio}")
print(f"Best alpha: {best_alpha}")
print(f"Training time: {training_time_en:.2f} seconds")


In [ ]:
from codecarbon import EmissionsTracker

tracker = EmissionsTracker(project_name="ElasticNet_Training")
tracker.start()

elastic_pipeline.fit(X_train, y_train)

training_emissions = tracker.stop()  # returns CO₂eq (kg)

print(f"Energy consumed: {tracker.final_emissions_data.energy_consumed:.6f} kWh")
print(f"CO₂ emissions: {training_emissions:.6f} kg CO₂eq")

tracker_pred = EmissionsTracker(project_name="ElasticNet_Prediction")
tracker_pred.start()

_ = elastic_pipeline.predict(X_test)

prediction_emissions = tracker_pred.stop()  # returns CO₂eq (kg)

# Compute normalized values
energy_per_1000 = tracker_pred.final_emissions_data.energy_consumed * (1000 / len(X_test))
co2_per_1000 = prediction_emissions * (1000 / len(X_test))

print(f"Energy consumed: {energy_per_1000:.8f} kWh")
print(f"CO₂ emissions: {co2_per_1000:.8f} kg CO₂eq")


In [ ]:

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Predictions
y_pred_en = elastic_pipeline.predict(X_test)

# Metrics
rmse_en = np.sqrt(mean_squared_error(y_test, y_pred_en))
mae_en = mean_absolute_error(y_test, y_pred_en)
r2_en = r2_score(y_test, y_pred_en)

print(f"\nElasticNet Model Results:")
print(f"Test RMSE: ${rmse_en:,.2f}")
print(f"Test MAE: ${mae_en:,.2f}")
print(f"Test R²: {r2_en:.4f}")

# Cross-validation
cv_scores_en = cross_val_score(elastic_pipeline, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
cv_rmse_en = np.sqrt(-cv_scores_en)

print(f"\nCross-Validation RMSE: {cv_rmse_en.mean():.2f} (+/- {cv_rmse_en.std() * 2:.2f})")


In [ ]:
elastic_model = elastic_pipeline.named_steps["elasticnet"]

coefs = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": elastic_model.coef_
}).sort_values(by="Coefficient", key=abs, ascending=False)

print("\nTop 10 Most Influential Features (by coefficient magnitude):")
print(coefs.head(10))


In [ ]:
plt.figure(figsize=(12, 5))

# Actual vs Predicted
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_en, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Salary")
plt.ylabel("Predicted Salary")
plt.title(f"ElasticNet: Actual vs Predicted\nR² = {r2_en:.4f}")

plt.subplot(1, 2, 2)
top_feats = coefs.head(10)
plt.barh(range(len(top_feats)), top_feats["Coefficient"])
plt.yticks(range(len(top_feats)), top_feats["Feature"])
plt.title("Top 10 ElasticNet Coefficients")

plt.tight_layout()
plt.show()


In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(xgb_best)

shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title('XGBoost SHAP Feature Importance (Bar Plot)')
plt.tight_layout()
plt.show()

shap.summary_plot(shap_values, X_test, show=False)
plt.title('XGBoost SHAP Feature Impact (Dot Plot)')
plt.tight_layout()
plt.show()


**Reasoning**:
To interpret the ElasticNet model using SHAP, I need to import the shap library, initialize a LinearExplainer, and compute SHAP values for the test set. ElasticNet is a linear model, so `shap.LinearExplainer` is the appropriate choice.



In [ ]:
import shap
import matplotlib.pyplot as plt

scaler = elastic_pipeline.named_steps['scaler']
X_test_scaled = scaler.transform(X_test)

elasticnet_model = elastic_pipeline.named_steps['elasticnet']

X_train_scaled = scaler.transform(X_train)
explainer_en = shap.LinearExplainer(elasticnet_model, X_train_scaled)

shap_values_en = explainer_en.shap_values(X_test_scaled)

X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

shap.summary_plot(shap_values_en, X_test_scaled_df, plot_type="bar", show=False)
plt.title('ElasticNet SHAP Feature Importance (Bar Plot)')
plt.tight_layout()
plt.show()

shap.summary_plot(shap_values_en, X_test_scaled_df, show=False)
plt.title('ElasticNet SHAP Feature Impact (Dot Plot)')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt

top_3_elastic_features = coefs['Feature'].head(3).tolist()

print(f"Top 3 ElasticNet features for PDPs: {top_3_elastic_features}")

fig, ax = plt.subplots(figsize=(15, 5), ncols=len(top_3_elastic_features))

# Use PartialDependenceDisplay.from_estimator
PartialDependenceDisplay.from_estimator(
    elastic_pipeline,
    X_train,
    features=top_3_elastic_features,
    feature_names=X_train.columns.tolist(), # Pass original feature names
    response_method='auto',
    ax=ax, # Pass the axes for subplots
    kind='average'
)

fig.suptitle('ElasticNet Partial Dependence Plots for Top 3 Features (Average Effect)')
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle overlap
plt.show()

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt

top_3_xgb_features = feature_importance['feature'].head(3).tolist()

print(f"Top 3 XGBoost features for PDPs: {top_3_xgb_features}")

fig, ax = plt.subplots(figsize=(15, 5), ncols=len(top_3_xgb_features))

PartialDependenceDisplay.from_estimator(
    xgb_best,
    X_train,
    features=top_3_xgb_features,
    feature_names=X_train.columns.tolist(),
    response_method='auto',
    ax=ax,
    kind='average'
)

fig.suptitle('XGBoost Partial Dependence Plots for Top 3 Features (Average Effect)')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()